<a href="https://colab.research.google.com/github/siddharth9238/Smart-door-lock-using-IoT-major-project/blob/main/behavioral_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**BEHAVIORAL MODEL BUILDING**

***PHASE 1: Load Dataset***

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving iot_behavior_dataset_realistic.csv to iot_behavior_dataset_realistic.csv


In [ ]:
import pandas as pd

file_name = list(uploaded.keys())[0]
df = pd.read_csv(file_name)

df.head()

In [ ]:
print(df.shape)
print(df.info())
print(df.describe())

***PHASE 2: Feature Engineering***

In [ ]:
# Convert hour into cyclic feature (VERY IMPORTANT)

import numpy as np

df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)

# Drop raw hour (optional but better)
df = df.drop(columns=["hour"])

df.head()

In [ ]:
features = ["hour_sin", "hour_cos", "day", "rssi", "distance"]

X = df[features]
y = df["label"]

***PHASE 3: Train-Test Split***

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

***PHASE 4: Handle Imbalance***

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)
weights = compute_class_weight(class_weight='balanced', classes=classes, y=y_train)

class_weights = dict(zip(classes, weights))
print(class_weights)

***PHASE 5: Train Advanced Model***

Gradient Descent is used for iterative Training

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

model = GradientBoostingClassifier(
    n_estimators=150,
    learning_rate=0.05,
    max_depth=5
)

model.fit(X_train, y_train)

***PHASE 6: Evaluation***

In [ ]:
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)

In [ ]:
from sklearn.metrics import accuracy_score

train_pred = model.predict(X_train)

print("Train Accuracy:", accuracy_score(y_train, train_pred))
print("Test Accuracy:", accuracy_score(y_test, y_pred))

NameError: name 'model' is not defined

In [ ]:
# Classification Report

from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

In [ ]:
# Confusion Matrix

from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

cm = confusion_matrix(y_test, y_pred)

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Confusion Matrix")
plt.show()

***PHASE 6: Decision Engine***

In [ ]:
def decision_engine(probs):
    normal, suspicious, attack = probs

    if attack > 0.5:
        return " ALERT (High Risk)"

    elif suspicious > 0.4:
        return " ASK OTP (Medium Risk)"

    else:
        return " UNLOCK (Safe)"

In [ ]:
def compute_risk_score(probs):
    normal, suspicious, attack = probs

    # Weighted risk calculation
    risk_score = (suspicious * 50) + (attack * 100)

    return round(risk_score, 2)

In [ ]:
def final_decision(probs):
    risk = compute_risk_score(probs)

    if risk >= 70:
        decision = " ALERT"

    elif risk >= 40:
        decision = " ASK OTP"

    else:
        decision = " UNLOCK"

    return {
        "risk_score": risk,
        "decision": decision,
        "probabilities": {
            "normal": probs[0],
            "suspicious": probs[1],
            "attack": probs[2]
        }
    }

In [ ]:
# Single Sample Test

sample = X_test.iloc[[0]]  # keep DataFrame

probs = model.predict_proba(sample)[0]

result = final_decision(probs)

print("Result:", result)

In [ ]:
# Multiple Sample Testing

for i in range(5):
    sample = X_test.iloc[[i]]
    probs = model.predict_proba(sample)[0]

    result = final_decision(probs)

    print(f"\n🔹 Sample {i}")
    print("Decision:", result["decision"])
    print("Risk Score:", result["risk_score"])

    print("Confidence:")
    print(f"  Normal: {probs[0]:.2f}")
    print(f"  Suspicious: {probs[1]:.2f}")
    print(f"  Attack: {probs[2]:.2f}")

In [ ]:
# Actual Labels vs Predicted Labels

for i in range(10):
    sample = X_test.iloc[[i]]
    true_label = y_test.iloc[i]

    probs = model.predict_proba(sample)[0]
    result = final_decision(probs)

    print(f"\nSample {i}")
    print("True Label:", true_label)
    print("Decision:", result["decision"])
    print("Risk Score:", result["risk_score"])

***PHASE 7: Adding a Layer of LSTM Model for Memorization***

In [ ]:
import numpy as np
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense
from sklearn.preprocessing import MinMaxScaler

In [ ]:
# Normalizing the data

features = ["hour_sin", "hour_cos", "day", "rssi", "distance"]

scaler = MinMaxScaler()
df_scaled = df.copy()
df_scaled[features] = scaler.fit_transform(df[features])

In [ ]:
# Creating Sequential Data

SEQUENCE_LENGTH = 5

def create_sequences(data, features):
    sequences = []

    for i in range(len(data) - SEQUENCE_LENGTH):
        seq = data[features].iloc[i:i+SEQUENCE_LENGTH].values
        sequences.append(seq)

    return np.array(sequences)

X_seq = create_sequences(df_scaled, features)

In [ ]:
# Shifing labels because sequence Predicts the Next Event

y_seq = df["label"].iloc[SEQUENCE_LENGTH:].values

In [ ]:
# Building LSTM Model

model_lstm = Sequential([
    LSTM(64, input_shape=(SEQUENCE_LENGTH, len(features))),
    Dense(32, activation='relu'),
    Dense(3, activation='softmax')   #  FIXED
])

model_lstm.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model_lstm.summary()

In [ ]:
# Get Predictions

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Predict probabilities
y_pred_probs = model_lstm.predict(X_test_seq)

# Convert to binary (threshold = 0.5)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
# Test-Train Split

split = int(0.8 * len(X_seq))

X_train_seq = X_seq[:split]
X_test_seq = X_seq[split:]

y_seq = df["label"].iloc[5:].values  # IMPORTANT alignment

y_train_seq = y_seq[:split]
y_test_seq = y_seq[split:]

In [ ]:
from sklearn.utils import class_weight
import numpy as np

weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(y_train_seq),
    y=y_train_seq
)

class_weights = dict(enumerate(weights))

In [ ]:
# Training LSTM Model

model_lstm.fit(
    X_train_seq,
    y_train_seq,
    epochs=15,
    batch_size=32,
    validation_data=(X_test_seq, y_test_seq),
    class_weight=class_weights
)

In [ ]:
# LSTM Prediction Function

def get_lstm_score(sequence):
    seq = np.expand_dims(sequence, axis=0)
    return model_lstm.predict(seq)[0][0]

In [ ]:
# Final Hybrid Prdiction Engine

def hybrid_prediction(sample, sequence):

    # Gradient Boosting score
    gb_probs = model.predict_proba(sample)[0]
    gb_score = (gb_probs[1] * 0.5) + (gb_probs[2] * 1.0)

    # LSTM score
    lstm_score = get_lstm_score(sequence)

    # Final combined score
    final_score = 0.6 * gb_score + 0.4 * lstm_score

    return final_score

In [ ]:
# Final Hybrid Decision Engine

def final_decision(final_score):

    if final_score < 0.3:
        return "OTP Required"

    elif final_score < 0.6:
        return "OTP + Biometric Required"

    else:
        return "Blocked + Alert Triggered"

In [ ]:
# Real-Time Simulation

sequence = df_scaled[features].iloc[-SEQUENCE_LENGTH:].values

sample = df.iloc[[-1]][features]

score = hybrid_prediction(sample, sequence)
decision = final_decision(score)

print("Final Score:", score)
print("Decision:", decision)

***PHASE 8: Final Evaluation Phase***

In [ ]:
# Get Predictions

from sklearn.metrics import confusion_matrix, classification_report, accuracy_score

# Predict probabilities
y_pred_probs = model_lstm.predict(X_test_seq)

# Convert to binary (threshold = 0.5)
y_pred = np.argmax(y_pred_probs, axis=1)

In [ ]:
# Accuracy, Precision, Recall and F1 Score

print("Accuracy:", accuracy_score(y_test_seq, y_pred))
print("\nClassification Report:\n")
print(classification_report(y_test_seq, y_pred))